In [2]:
from collections import defaultdict
# --------------------------------------------
# 1. SIMPLE TOKENIZER (from scratch)
# --------------------------------------------
def tokenize(text):
    # Convert text to lowercase for uniformity
    # Remove full stops (.)
    # Split sentence into words based on space
    return text.lower().replace(".", "").split()


In [3]:
# --------------------------------------------
# 2. BUILD N-GRAM COUNTS
# --------------------------------------------
def build_ngram_counts(tokens, n):
    # Dictionary to store frequency of n-grams
    counts = defaultdict(int)

    # Loop through tokens to create n-grams
    for i in range(len(tokens) - n + 1):
        # Extract n consecutive words
        ngram = tuple(tokens[i:i+n])  # Example: ('i','love')

        # Increase count of this n-gram
        counts[ngram] += 1

    return counts


In [4]:
# --------------------------------------------
# 3. UNIGRAM MODEL
# --------------------------------------------
def unigram_prob(word, unigram_counts, total_words):
    # Formula:
    # P(word) = Count(word) / Total number of words

    # get() returns 0 if word is not found
    return unigram_counts.get((word,), 0) / total_words


In [5]:
# --------------------------------------------
# 4. BIGRAM MODEL
# --------------------------------------------
def bigram_prob(w1, w2, bigram_counts, unigram_counts):
    # Formula:
    # P(w2 | w1) = Count(w1, w2) / Count(w1)

    # If first word never appears → probability = 0
    if unigram_counts.get((w1,), 0) == 0:
        return 0

    # Conditional probability calculation
    return bigram_counts.get((w1, w2), 0) / unigram_counts.get((w1,), 1)


In [6]:
# --------------------------------------------
# 5. TRIGRAM MODEL
# --------------------------------------------
def trigram_prob(w1, w2, w3, trigram_counts, bigram_counts):
    # Formula:
    # P(w3 | w1, w2) = Count(w1, w2, w3) / Count(w1, w2)

    # If bigram does not exist → probability = 0
    if bigram_counts.get((w1, w2), 0) == 0:
        return 0

    return trigram_counts.get((w1, w2, w3), 0) / bigram_counts.get((w1, w2), 1)


In [7]:
# --------------------------------------------
# 6. SENTENCE PROBABILITY (BIGRAM MODEL)
# --------------------------------------------
def sentence_probability(sentence, bigram_counts, unigram_counts):
    # Tokenize input sentence
    tokens = tokenize(sentence)

    # Initialize probability to 1 (multiplicative rule)
    prob = 1

    # Loop through word pairs
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i+1]

        # Get bigram probability
        p = bigram_prob(w1, w2, bigram_counts, unigram_counts)

        # Multiply probabilities
        prob *= p

    return prob

In [8]:
# --------------------------------------------
# MAIN PROGRAM
# --------------------------------------------
if __name__ == "__main__":

    # Input corpus
    text = "I love NLP and I love machine learning"

    # Step 1: Tokenization
    tokens = tokenize(text)
    print("Tokens:", tokens)

    # Step 2: Build N-gram counts
    unigram_counts = build_ngram_counts(tokens, 1)
    bigram_counts = build_ngram_counts(tokens, 2)
    trigram_counts = build_ngram_counts(tokens, 3)

    # Total number of words
    total_words = len(tokens)

    # Display counts
    print("\nUnigram Counts:", dict(unigram_counts))
    print("\nBigram Counts:", dict(bigram_counts))
    print("\nTrigram Counts:", dict(trigram_counts))

    # ----------------------------------------
    # TEST PROBABILITIES
    # ----------------------------------------
    print("\n--- Probabilities ---")

    # Example: Unigram probability
    print("P(love) =", unigram_prob("love", unigram_counts, total_words))

    # Example: Bigram probability
    print("P(love | i) =", bigram_prob("i", "love", bigram_counts, unigram_counts))

    # Example: Trigram probability
    print("P(learning | machine, learning) =",
          trigram_prob("machine", "learning", "learning", trigram_counts, bigram_counts))

    # ----------------------------------------
    # SENTENCE PROBABILITY
    # ----------------------------------------
    sentence = "I love machine learning"

    # Compute probability using bigram model
    prob = sentence_probability(sentence, bigram_counts, unigram_counts)

    print("\nSentence:", sentence)
    print("Sentence Probability (Bigram):", prob)


Tokens: ['i', 'love', 'nlp', 'and', 'i', 'love', 'machine', 'learning']

Unigram Counts: {('i',): 2, ('love',): 2, ('nlp',): 1, ('and',): 1, ('machine',): 1, ('learning',): 1}

Bigram Counts: {('i', 'love'): 2, ('love', 'nlp'): 1, ('nlp', 'and'): 1, ('and', 'i'): 1, ('love', 'machine'): 1, ('machine', 'learning'): 1}

Trigram Counts: {('i', 'love', 'nlp'): 1, ('love', 'nlp', 'and'): 1, ('nlp', 'and', 'i'): 1, ('and', 'i', 'love'): 1, ('i', 'love', 'machine'): 1, ('love', 'machine', 'learning'): 1}

--- Probabilities ---
P(love) = 0.25
P(love | i) = 1.0
P(learning | machine, learning) = 0.0

Sentence: I love machine learning
Sentence Probability (Bigram): 0.5
